## Step 1: Load the Labeled Table
Load the artifact from Notebook 2.

In [1]:
import pandas as pd

labeled_table = pd.read_csv("artifacts/labeled_table.csv", parse_dates=["order_purchase_timestamp"])

print(f"Loaded labeled_table: {labeled_table.shape[0]} rows, {labeled_table.shape[1]} columns")
print(f"Date range: {labeled_table['order_purchase_timestamp'].min()} to {labeled_table['order_purchase_timestamp'].max()}")

Loaded labeled_table: 99441 rows, 21 columns
Date range: 2016-09-04 21:15:19 to 2018-10-17 17:30:18


## Step 2: Time-Based Split
Split the data by `order_purchase_timestamp`: the oldest 70% of orders go to train, the next 15% to validation, and the most recent 15% to test. This mimics how the model will be used in production — trained on the past, evaluated on the future.

In [2]:
# Sort by purchase date
labeled_table = labeled_table.sort_values("order_purchase_timestamp").reset_index(drop=True)

# Compute cutoff points
n = len(labeled_table)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = labeled_table.iloc[:train_end]
val = labeled_table.iloc[train_end:val_end]
test = labeled_table.iloc[val_end:]

print(f"Train: {len(train)} rows | {train['order_purchase_timestamp'].min()} to {train['order_purchase_timestamp'].max()}")
print(f"Val:   {len(val)} rows | {val['order_purchase_timestamp'].min()} to {val['order_purchase_timestamp'].max()}")
print(f"Test:  {len(test)} rows | {test['order_purchase_timestamp'].min()} to {test['order_purchase_timestamp'].max()}")

Train: 69608 rows | 2016-09-04 21:15:19 to 2018-04-13 21:50:49
Val:   14916 rows | 2018-04-13 21:53:34 to 2018-06-20 16:28:00
Test:  14917 rows | 2018-06-20 16:28:15 to 2018-10-17 17:30:18


## Step 3: Check Label Balance Across Splits
Verify that the class imbalance (late vs on-time) is roughly consistent across train, validation, and test — this confirms the time-based split didn't introduce a very different distribution in any split.

In [3]:
for name, split in [("Train", train), ("Val", val), ("Test", test)]:
    pct = split["is_late"].value_counts(normalize=True, dropna=True) * 100
    print(f"{name}: on-time={pct.get(0, 0):.2f}%, late={pct.get(1, 0):.2f}%")

Train: on-time=90.95%, late=9.05%
Val: on-time=94.64%, late=5.36%
Test: on-time=93.43%, late=6.57%


## Step 4: Save Train, Validation, and Test Artifacts
Save each split as a separate CSV file for the next notebooks.

**Note:** The late-delivery rate varies across splits (9.05% train, 5.36% val, 6.57% test). This is expected with a time-based split and may reflect real changes in delivery performance over time — worth investigating further in the EDA notebook.

In [4]:
train.to_csv("artifacts/train.csv", index=False)
val.to_csv("artifacts/val.csv", index=False)
test.to_csv("artifacts/test.csv", index=False)

print("✅ Saved train.csv, val.csv, and test.csv")

✅ Saved train.csv, val.csv, and test.csv
